# 02 — Training Demo

- Visualise augmentations (before/after)
- Model architecture summary
- Learning rate schedule plot
- Short proof-of-concept training run (3 epochs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch

!pip install -q timm albumentations torchinfo

DRIVE_ROOT = Path('/content/drive/MyDrive/AI_TRAINING/skinDisease')
LOCAL_DATA  = Path('/content/data')

if LOCAL_DATA.exists():
    print('Data already on VM — skipping copy.')
else:
    print('Copying data from Drive...')
    shutil.copytree(str(DRIVE_ROOT / 'data/raw'), str(LOCAL_DATA))
    print('Done!')

IMAGE_SIZE  = 224
NUM_CLASSES = 23
BACKBONE    = 'efficientnet_b0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Augmentation Visualisation

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_train_transforms(size):
    return A.Compose([
        A.RandomResizedCrop(size, size, scale=(0.7, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05, p=0.5),
        A.ElasticTransform(alpha=60, sigma=6, p=0.3),
        A.GaussNoise(p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

data_dir = LOCAL_DATA
sample_img_path = next(data_dir.rglob('*.[jJpP][pPnN][gG]'), None)

if sample_img_path:
    img = cv2.cvtColor(cv2.imread(str(sample_img_path)), cv2.COLOR_BGR2RGB)
    train_tf = get_train_transforms(IMAGE_SIZE)

    fig, axes = plt.subplots(2, 6, figsize=(18, 6))
    orig_resized = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    for col in range(6):
        axes[0, col].imshow(orig_resized)
        axes[0, col].axis('off')
        axes[0, col].set_title('Original', fontsize=8)

    for col in range(6):
        aug_tensor = train_tf(image=img)['image']
        aug_np = np.clip(aug_tensor.permute(1,2,0).numpy() * STD + MEAN, 0, 1)
        axes[1, col].imshow(aug_np)
        axes[1, col].axis('off')
        axes[1, col].set_title(f'Augmented #{col+1}', fontsize=8)

    plt.suptitle('Train Augmentation Variations', fontsize=13)
    plt.tight_layout()
    fig_dir = DRIVE_ROOT / 'outputs' / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_dir / 'augmentation_demo.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No images found — check your Drive path.')

## 2. Model Architecture Summary

In [ ]:
import timm
from torchinfo import summary

def build_model(backbone, num_classes):
    model = timm.create_model(backbone, pretrained=False, num_classes=num_classes)
    return model

model = build_model(BACKBONE, NUM_CLASSES)
summary(model, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE),
        col_names=['input_size', 'output_size', 'num_params', 'trainable'])

## 3. Learning Rate Schedule

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

EPOCHS       = 40
LR           = 3e-4
WARMUP_EPOCHS = 3
MIN_LR       = 1e-6

model_lr = build_model(BACKBONE, NUM_CLASSES)
opt = AdamW(model_lr.parameters(), lr=LR)
warmup = LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(opt, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=MIN_LR)
scheduler = SequentialLR(opt, [warmup, cosine], milestones=[WARMUP_EPOCHS])

lrs = []
for _ in range(EPOCHS):
    lrs.append(opt.param_groups[0]['lr'])
    scheduler.step()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, EPOCHS+1), lrs, linewidth=2, color='steelblue')
ax.axvline(WARMUP_EPOCHS, color='red', linestyle='--',
           label=f'Phase 2 start (epoch {WARMUP_EPOCHS+1})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Learning Rate')
ax.set_title('LR Schedule: Linear Warmup + Cosine Annealing')
ax.set_yscale('log'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_ROOT / 'outputs' / 'figures' / 'lr_schedule_preview.png', dpi=150)
plt.show()

## 4. Quick Smoke Test (1 batch)

In [ ]:
proc = DRIVE_ROOT / 'data' / 'processed'
if (proc / 'train.csv').exists():
    from torch.utils.data import Dataset, DataLoader
    from albumentations.pytorch import ToTensorV2
    import albumentations as A

    class SkinDataset(Dataset):
        def __init__(self, df, transform):
            self.df = df.reset_index(drop=True)
            self.transform = transform
        def __len__(self): return len(self.df)
        def __getitem__(self, i):
            row = self.df.iloc[i]
            img = cv2.cvtColor(cv2.imread(row['path']), cv2.COLOR_BGR2RGB)
            img = self.transform(image=img)['image']
            return img, int(row['label'])

    val_tf = A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

    train_df = pd.read_csv(proc / 'train.csv')
    val_df   = pd.read_csv(proc / 'val.csv')
    loader   = DataLoader(SkinDataset(train_df, val_tf), batch_size=8, shuffle=True)

    imgs, labels = next(iter(loader))
    model_test = build_model(BACKBONE, NUM_CLASSES).to(device)
    with torch.no_grad():
        logits = model_test(imgs.to(device))
    print(f'Input shape : {imgs.shape}')
    print(f'Output shape: {logits.shape}  (batch x {NUM_CLASSES} classes)')
    print('Forward pass OK!')
else:
    print('No processed CSVs found — run scripts/prepare_data.py first.')